In [1]:
from transformers import BertTokenizer, BertModel, RobertaTokenizer, RobertaModel, LongformerModel, LongformerTokenizer, BertConfig
import torch.nn as nn
import torch
from json import loads
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score
from neuralNetwork import NeuralNetwork
from prettytable import PrettyTable

# Model definition

In [2]:
class Model(NeuralNetwork):
    def __init__(self, base_model_name:str, num_classes:int, dropout:float = .5) -> None:
        super().__init__()
        if "FacebookAI/roberta-base" == base_model_name:
            self.bert = RobertaModel.from_pretrained(base_model_name)
        elif "bert-base-uncased" == base_model_name:
            self.bert = BertModel.from_pretrained(base_model_name)
        elif "allenai/longformer-base-4096":
            self.bert = LongformerModel.from_pretrained("allenai/longformer-base-4096")
        else:
            raise Exception("bert type unrecognised")
        self.dropout = nn.Dropout(dropout)

        # self.conv1 = nn.Conv1d(self.bert.config.hidden_size * 2, self.bert.config.hidden_size * 2, 1, 1)

        # self.pool = nn.AvgPool1d(3, 2)
        # self.conv2 = nn.Conv1d(767, 767, 1, 1)
        # self.conv3 = nn.Conv1d(383, 383, 1, 1)
        
        # self.trainsfomation_combined = nn.Linear(191, 100)
        # self.trainsfomation_combined2 = nn.Linear(100, 50)

        self.output_layer = nn.Linear(self.bert.config.hidden_size, num_classes)

        self.relu = nn.ReLU()

    def forward(self, inputs):
        # model, instance, times = inputs['model'], inputs['instance'], inputs["times"]
        # _, encoded_instance = self.bert(**instance, return_dict = False)
        # _, encoded_model = self.bert(**model, return_dict = False)
        # encoded_instance = self.dropout(encoded_instance)
        # encoded_model = self.dropout(encoded_model)

        # encoded_input = torch.cat((encoded_model, encoded_instance, times), dim=1)
        _, encoded_input = self.bert(**inputs, return_dict = False)
        encoded_input = self.dropout(encoded_input)



        # size = encode_input.size()
        # encoded_input = encoded_input.reshape(size[0], size[1], -1)
        # out = self.conv1(encoded_input)
        # out = self.relu(out)
        # size = out.size()
        # out = out.reshape(size[0], size[1])
        # out = self.pool(out)
        # size = out.size()

        # out = out.reshape(size[0], size[1], -1)
        # out = self.conv2(out)
        # out = self.relu(out)
        # size = out.size()
        # out = out.reshape(size[0], size[1])
        # out = self.pool(out)
        # size = out.size()

        # out = out.reshape(size[0], size[1], -1)
        # out = self.conv3(out)
        # out = self.relu(out)
        # size = out.size()
        # out = out.reshape(size[0], size[1])
        # out = self.pool(out)

        # out = self.trainsfomation_combined(out)     
        # out = self.trainsfomation_combined2(out)   
        
        return self.output_layer(encoded_input)
    
class My_encoder_model(NeuralNetwork):
    def __init__(self, num_classes:int, dropout:float = .5) -> None:
        super().__init__()

        self.config = BertConfig(max_position_embeddings=2567)

        self.encoder = BertModel(self.config)
        self.dropout = nn.Dropout(dropout)

        self.output_layer = nn.Linear(self.encoder.config.hidden_size, num_classes)

        self.relu = nn.ReLU()

    def forward(self, inputs):
        _, encoded_input = self.encoder(**inputs, return_dict = False)
        encoded_input = self.dropout(encoded_input)
        
        return self.output_layer(encoded_input)
    

def get_tokenizer(bert_type:str):
    if "FacebookAI/roberta-base" == bert_type:
        return RobertaTokenizer.from_pretrained(bert_type)
    elif "bert-base-uncased" == bert_type:
        return BertTokenizer.from_pretrained(bert_type)
    elif "allenai/longformer-base-4096" == bert_type:
        return LongformerTokenizer.from_pretrained("allenai/longformer-base-4096")
    else:
        raise Exception("bert type unrecognised")
    
def count_parameters(model):
    table = PrettyTable(["Modules", "Parameters"])
    total_params = 0
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        params = parameter.numel()
        table.add_row([name, params])
        total_params += params
    print(table)
    print(f"Total Trainable Params: {total_params}")
    return total_params

# Data import

In [3]:
f = open("./data/datasets/dataset_CarSequencing-2024-03-02.json")
data = loads(f.read())
f.close()

In [4]:
problem_specification = """
$ A number of cars are to be produced; they are not identical, because different options are available as variants on the basic model. The assembly line has different stations which install the various options (air-conditioning, sun-roof, etc.). These stations have been designed to handle at most a certain percentage of the cars passing along the assembly line. Furthermore, the cars requiring a certain option must not be bunched together, otherwise the station will not be able to cope. Consequently, the cars must be arranged in a sequence so that the capacity of each station is never exceeded. For instance, if a particular station can only cope with at most half of the cars passing along the line, the sequence must be built so that at most 1 car in any 2 requires that option. The problem has been shown to be NP-complete (Gent 1999).
$ The format of the data files is as follows:
$     First line: number of cars; number of options; number of classes.
$     Second line: for each option, the maximum number of cars with that option in a block.
$     Third line: for each option, the block size to which the maximum number refers.
$     Then for each class: index no.; no. of cars in this class; for each option, whether or not this class requires it (1 or 0). 
$ This is the example given in (Dincbas et al., ECAI88):
$ 10 5 6
$ 1 2 1 2 1
$ 2 3 3 5 5
$ 0 1 1 0 1 1 0 
$ 1 1 0 0 0 1 0 
$ 2 2 0 1 0 0 1 
$ 3 2 0 1 0 1 0 
$ 4 2 1 0 1 0 0 
$ 5 2 1 1 0 0 0 
$ A valid sequence for this set of cars is:
$ Class 	Options req.
$ 0 	1 0 1 1 0
$ 1 	0 0 0 1 0
$ 5 	1 1 0 0 0
$ 2 	0 1 0 0 1
$ 4 	1 0 1 0 0
$ 3 	0 1 0 1 0
$ 3 	0 1 0 1 0
$ 4 	1 0 1 0 0
$ 2 	0 1 0 0 1
$ 5 	1 1 0 0 0
"""

model_string = """
given n_cars, n_classes, n_options : int(1..)
letting Slots  be domain int(1..n_cars),
        Class  be domain int(1..n_classes),
        Option be domain int(1..n_options),
given quantity      : function (total) Class  --> int(1..),
      maxcars       : function (total) Option --> int(1..),
      blksize_delta : function (total) Option --> int(1..),
      usage         : relation (minSize 1) of ( Class * Option )
$ There must be at least as many cars as there are n_classes as quantity is indexed from 1..
where n_cars >= n_classes
$ The sum of the cars in the quantity function should equal n_cars 
where ( sum quant : Class . quantity(quant) ) = n_cars
$ Blksize must be greater than maxcars for all options
$ where forAll option: Option . maxcars(option) < blksize(option)
$ Make sure that all options are used at least once
where  forAll option: Option .  |toSet(usage(_,option))| >= 1
$ Make sure that all classes have at least one option
where  forAll class: Class .  |toSet(usage(class,_))| >= 1
find car : function (total) Slots --> Class
such that
    forAll c : Class . |preImage(car,c)| = quantity(c),
    forAll opt : Option .
        forAll s : int(1..n_cars+1-(maxcars(opt)+blksize_delta(opt))) .
            (sum i : int(s..s+(maxcars(opt)+blksize_delta(opt))-1) . toInt(usage(car(i),opt))) <= maxcars(opt)"""

model_input = f"{problem_specification}\n\n{model_string}"

In [5]:
def one_hot_encoding(data, values):
    length = len(values)

    out = []

    for datapoint in data:
        tensor = torch.zeros(size=(length, ))

        idx = values.index(datapoint)
        tensor[idx] = 1.
        out.append(tensor)

    return out

# Data manipulation

In [6]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [7]:
BERT_TYPE = "bert-base-uncased"

In [8]:
tokenizer = get_tokenizer(BERT_TYPE)

In [9]:
all_times = [datapoint["all_times"] for datapoint in data]

In [10]:
y = [datapoint["combination"] for datapoint in data]
combinations = [d["combination"] for d in data[0]["all_times"]]
y = one_hot_encoding(y, combinations)

In [11]:
all_times_matrix = np.zeros(np.array(y).shape)
for i in range(len(all_times)):
    times = all_times[i]
    for j in range(len(times)):
        all_times_matrix[i,j] = times[j]["time"]

In [12]:
def dict_lists_to_list_of_dicts(input_dict:dict) -> list[dict]:
    """
    A function that convert a dictionary of lists into a list of dictionaries
    Parameters
    ----------
    input_dict:dict
        The dictionary to convert
    
    Outputs
    -------
    The list of dictionaries
    """
    keys = input_dict.keys()
    list_lengths = [len(input_dict[key]) for key in keys]

    if len(set(list_lengths)) > 1:
        raise ValueError("All lists in the input dictionary must have the same length.")

    list_of_dicts = [{key: input_dict[key][i] for key in keys} for i in range(list_lengths[0])]

    return list_of_dicts

In [13]:
model_input_tokens = tokenizer(model_input, padding=True, truncation=True, return_tensors='pt')
model_input_tokens = {
        key: model_input_tokens[key].reshape(-1) for key in model_input_tokens.keys()
    }

instances_and_model = [f"{model_input}\n\n{datapoint['instance']}" for datapoint in data]
# tokenized_instances = tokenizer([datapoint["instance"] for datapoint in data], padding=True, truncation=True, return_tensors='pt') 
# keys = tokenized_instances.keys()
# x = [{
#     "instance": {
#         key: tokenized_instances[key][i] for key in keys
#     },
#     "model": model_input_tokens,
#     "mean": torch.from_numpy(all_times_matrix[i,:])
# } for i in range(len(tokenized_instances["input_ids"]))]

x = dict_lists_to_list_of_dicts(tokenizer(instances_and_model, padding=True, return_tensors='pt'))

Token indices sequence length is longer than the specified maximum sequence length for this model (1340 > 512). Running this sequence through the model will result in indexing errors


In [14]:
BUCKETS = 10

N_ELEMENTS = len(x)

BUCKET_SIZE = N_ELEMENTS // BUCKETS

TRAIN_VALIDATION_BUCKETS = 9
TEST_BUCKETS = 1

TRAIN_SIZE = int((TRAIN_VALIDATION_BUCKETS / 10) * 9)
VALIDATION_SIZE = TRAIN_VALIDATION_BUCKETS - TRAIN_SIZE

buckets_x = [x[bucket * BUCKET_SIZE: (bucket + 1) * BUCKET_SIZE] for bucket in range(BUCKETS)]
buckets_y = [y[bucket * BUCKET_SIZE: (bucket + 1) * BUCKET_SIZE] for bucket in range(BUCKETS)]

x_train_validation = buckets_x[:TRAIN_VALIDATION_BUCKETS]
y_train_validation = buckets_y[:TRAIN_VALIDATION_BUCKETS]


x_train, y_train = np.ravel(x_train_validation[:TRAIN_SIZE]), np.ravel(y_train_validation[:TRAIN_SIZE])
x_validation, y_validation = np.ravel(x_train_validation[TRAIN_SIZE:TRAIN_SIZE + VALIDATION_SIZE]), np.ravel(y_train_validation[TRAIN_SIZE:TRAIN_SIZE + VALIDATION_SIZE])
x_test, y_test = np.ravel(buckets_x[-TEST_BUCKETS:]), np.ravel(buckets_y[-TEST_BUCKETS:])

In [15]:
BATCH_SIZE = 4
train_dataset = Dataset(x_train, y_train)
validation_dataset = Dataset(x_validation, y_validation)
test_dataset = Dataset(x_test, y_test)
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=BATCH_SIZE)
test_dataloader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
unique_combinations = combinations

# Model training

In [16]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("operating on device:", device)

operating on device: cuda:0


In [17]:
def extraction_function(x:torch.Tensor):
    return torch.max(x, -1)[1].cpu()

In [18]:
running_combinations = [d["combination"] for d in data]
frequencies = [running_combinations.count(c) for c in unique_combinations]
max_freq = max(frequencies)
min_freq = min(frequencies)
norm_factor = max_freq - min_freq 
weights = [1 + ((max_freq - freq)/norm_factor) for freq in frequencies]
weights = torch.Tensor(weights)
#weights = weights.to(device)

In [19]:
class Distance:
    def __init__(self, running_statistics:np.ndarray, batch_size:int) -> None:
        self.stats = running_statistics
        self.batch_size = batch_size
        self.__compute_weights()
    
    def compute(self, y_predict:torch.Tensor, y_true:torch.Tensor, batch_idx) -> float:
        low, high = batch_idx * self.batch_size, (batch_idx + 1) * (self.batch_size)
        weights = self.weights[low:high, :]
        return np.sum([weights[i, y_predict[i]] for i in range(y_predict.size()[0])]) / self.batch_size

    def __compute_weights(self) -> None:
        self.weights = np.zeros(shape=self.stats.shape)
        rows, columns = self.stats.shape
        l = 1e-07
        
        for i in range(rows):
            row = self.stats[i,:]
            max_row = np.max(row)
            min_row = np.min(row)
            norm_factor = max_row - min_row + l
            for j in range(columns):
                self.weights[i,j] = 1 - (row[j] - min_row) / norm_factor

def accuracy(y_predict:torch.Tensor, y_true:torch.Tensor, batch_idx) -> float:
    return accuracy_score(y_true, y_predict)

In [20]:
distance = Distance(all_times_matrix, 4)

In [21]:
length = len(unique_combinations)
#model = Model(BERT_TYPE, length, dropout=.3)
model = My_encoder_model(length, .2)

In [22]:
model.train_network(train_dataloader, 
                    validation_dataloader, 
                    torch.optim.SGD, 
                    loss_function=nn.CrossEntropyLoss(),
                    device=device, 
                    verbose=True, 
                    output_extraction_function=extraction_function, 
                    metrics={"accuracy": accuracy},
                    learning_rate=1e-03,
                    epochs=5);

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.18 GiB. GPU 0 has a total capacty of 7.78 GiB of which 630.62 MiB is free. Process 14893 has 7.15 GiB memory in use. Of the allocated memory 6.42 GiB is allocated by PyTorch, and 576.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [28]:
def __to(data, device):
  if isinstance(data, dict):
    return {key: __to(data[key], device) for key in data.keys()}
  elif isinstance(data, list):
    return [__to(d, device) for d in data]
  elif isinstance(data, tuple):
    return tuple([__to(d, device) for d in data])
  else:
    return data.to(device)
  
def __remove(data):
  if isinstance(data, dict):
    for key in data.keys():
      __remove(data[key])
  elif isinstance(data, list) or isinstance(data, tuple):
    for d in data:
      __remove(d)
  else:
    del data

In [47]:
all_prediction = []

In [48]:
model = model.to(device)

In [49]:
with torch.no_grad():
    for batch_idx, data in enumerate(train_dataloader):
      labels = data[1]
      inputs = data[0]
      if True:
        inputs = __to(data[0], device)
      outputs = model(inputs)
      predicted_classes = extraction_function(outputs)
      all_prediction += predicted_classes
      if True:
        __remove(inputs)
        del labels
        torch.cuda.empty_cache()

In [ ]:
model.load_state_dict(torch.load("./weights"))

<All keys matched successfully>

In [56]:
torch.save(model.state_dict(), "./data/weights/roberta_.3_all_stats")

In [43]:
y[:600]
y_x = np.array(y[:600])
values = [np.sum(y_x[:,i]) for i in range(8)]
argmax = np.argmax(values)
y_majority = np.zeros(shape=y_x.shape)
y_majority[:,argmax] = 1
print(accuracy_score(y_x, y_majority))
distance.batch_size = 600
y_majority_tensor = torch.tensor([argmax for _ in range(600)]) 
print(distance.compute(y_majority_tensor, y_majority_tensor, 0))

0.2966666666666667
0.9890348397347046


In [50]:
argmax_val = [all_times_matrix[i, argmax] for i in range(600)]
min_val = [min(all_times_matrix[i, :]) for i in range(600)]

sum(argmax_val), sum(min_val)

(4030.484999999999, 788.2039999999995)

In [51]:
pred_val = [all_times_matrix[i, all_prediction[i]] for i in range(600)]

In [52]:
sum(pred_val)

3994.9749999999995

In [ ]:
from random import randint

In [41]:
y_random = np.zeros(shape=y_x.shape)
for i in range(y_x.shape[0]):
    j = randint(0, 7)
    y_random[i,j] = 1
accuracy_score(y_x, y_random)

NameError: name 'randint' is not defined

In [ ]:
model.train_network(train_dataloader, 
                    validation_dataloader, 
                    torch.optim.SGD, 
                    device='cuda', 
                    verbose=True, 
                    output_extraction_function=extraction_function, 
                    metrics={"accuracy": accuracy_score},
                    learning_rate=1e-07,
                    epochs=20);

EPOCH 1 training loss:      1.138 - validation loss:      1.156                                                                                                                                                                              
EPOCH 1 training accuracy:      0.449 - validation accuracy:      0.390
----------------------------------------------------------------------------------------------------

EPOCH 2 training loss:      1.138 - validation loss:      1.156                                                                                                                                                                              
EPOCH 2 training accuracy:      0.449 - validation accuracy:      0.390
----------------------------------------------------------------------------------------------------

EPOCH 3 training loss:      1.141 - validation loss:      1.156                                                                                                                 